# ChronoAlign example

This notebook demonstrates `chronoalign` 0.2.0 from environment verification through a small end-to-end circadian alignment workflow. It is designed to run top-to-bottom in the active VS Code Python kernel.

## 1. Verify the active Python environment

The notebook uses the kernel's interpreter for installation and imports.

In [1]:
import platform
import sys

print("Python executable:", sys.executable)
print("Python version:", platform.python_version())
print("Platform:", platform.platform())

Python executable: /Users/arahjou/Documents/Com_Conda/.conda/bin/python
Python version: 3.11.8
Platform: macOS-26.6.2-arm64-arm-64bit


## 2. Install or upgrade the package in the active environment

This command targets the current notebook kernel. The local editable package is already available in the requested environment, so pip will confirm or update that installation.

In [2]:
import sys

package_name = "chronoalign"
!{sys.executable} -m pip install -U {package_name}

## 3. Validate the installation

In [3]:
import importlib
from importlib.metadata import version

chronoalign_version = version(package_name)
chrono = importlib.import_module(package_name)
print(f"{package_name} version: {chronoalign_version}")

chronoalign version: 0.2.0


## 4. Minimal usage example

A sleep-only fit creates a Route A profile and can produce a phase-sampling plan.

In [4]:
import numpy as np
import pandas as pd

from chronoalign.synthetic import SimConfig, simulate_observation, simulate_participant

TZ = "Europe/Berlin"
sim = simulate_participant(seed=7, cfg=SimConfig(n_days=14))
sleep = sim["sleep"]
truth = sim["truth"]

route_a = chrono.fit(sleep=sleep, tz=TZ)
print(route_a.summary())
print("\nSampling plan:")
print(chrono.schedule_phase_sample(assay="HairTime", prior=route_a))

CircadianProfile  tz=Europe/Berlin  route=A  grade=S1 (Sleep + alarm/work/free-day (shift type))
  behavioural      MSFsc 03:19 (SE 16 min)
  sleep            n=14 free=4 SJL=1.05 h SRI=90 schedule=regular

Sampling plan:
{'assay': 'HairTime', 'prior': 'sleep-based: MSFsc 03:19 - 6 h (SD 1 h, population phase-angle assumption)', 'validated_window_after_dlmo': (9.0, 17.0), 'robust_window': ('08:19', '12:19'), 'recommended_time': '10:19', 'prob_in_window': 1.0, 'note': 'record exact sampling time, timezone and day type; avoid evening and night samples'}


## 5. End-to-end example with synthetic inputs

Add one HairTime observation, fit the dynamic Route C profile, transform a CGM-like series, inspect identifiability diagnostics, and generate a wake-anchored schedule.

In [5]:
obs = simulate_observation(truth, 8, rng=np.random.default_rng(1))
obs.qc = {"model_sd_h": 0.30}
route_c = chrono.fit(
    sleep=sleep,
    phase_observations=[obs],
    tz=TZ,
    dynamic=True,
)
print(route_c.summary())

CircadianProfile  tz=Europe/Berlin  route=C  grade=C (Phase observation(s) + sleep (+/- light/activity))
  phase anchor     21:25 pDLMO[HairTime]  date=2026-03-10 (evening); sample day type: workday  SD=1.49 h  mode=smoother
  behavioural      MSFsc 03:19 (SE 16 min)
  phase angle      onset +1.32 h, midsleep +5.17 h, wake +9.02 h (relative to DLMO on the anchor evening)
  sleep            n=14 free=4 SJL=1.05 h SRI=90 schedule=regular
  observation      pDLMO[HairTime] 21:25 (evening 2026-03-10) SD=1.49 h  [model disagreement 0.30 h added to uncertainty]


In [6]:
t = pd.date_range(
    "2026-03-09 00:00",
    "2026-03-11 00:00",
    freq="15min",
    tz=TZ,
)
cgm = pd.DataFrame(
    {
        "datetime": t,
        "glucose": 95 + 8 * np.sin(2 * np.pi * (t.hour + t.minute / 60) / 24),
    }
)
aligned = chrono.transform(
    cgm,
    route_c,
    reference=["phase", "chronotype", "wake"],
    circular_encoding=True,
)
aligned.head()

,datetime,glucose,clock_time_h,asleep,hours_since_wake,prior_sleep_duration_h,crt_phase_h,crt_linear_h,crt_sin,crt_cos,pdlmo_phase_h,pdlmo_linear_h,phase_uncertainty_h,anchor_source,phase_mode,pdlmo_sin,pdlmo_cos
0,2026-03-09 00:00:00+01:00,95.000000,0.00,True,NaN,NaN,20.682876,20.682876,-0.763310,0.646033,2.656299,2.656299,1.458912,pDLMO[HairTime],smoother,0.640706,0.767786
1,2026-03-09 00:15:00+01:00,95.523225,0.25,True,NaN,NaN,20.932876,20.932876,-0.719423,0.694572,2.906299,2.906299,1.458912,pDLMO[HairTime],smoother,0.689550,0.724238
2,2026-03-09 00:30:00+01:00,96.044210,0.50,True,NaN,NaN,21.182876,21.182876,-0.672456,0.740138,3.156299,3.156299,1.458912,pDLMO[HairTime],smoother,0.735441,0.677589
3,2026-03-09 00:45:00+01:00,96.560723,0.75,True,NaN,NaN,21.432876,21.432876,-0.622608,0.782534,3.406299,3.406299,1.458912,pDLMO[HairTime],smoother,0.778183,0.628038
4,2026-03-09 01:00:00+01:00,97.070552,1.00,True,NaN,NaN,21.682876,21.682876,-0.570095,0.821579,3.656299,3.656299,1.458912,pDLMO[HairTime],smoother,0.817592,0.575798


In [7]:
print("Psi diagnostics:")
print(chrono.psi_diagnostics(aligned))

wake_schedule = chrono.schedule(
    route_c,
    anchor="wake",
    offsets=[2, 7, 12],
    dates=["2026-03-10", "2026-03-14"],
    mode="predicted",
    last_before_sleep=1,
)
wake_schedule[["date", "day_type", "clock", "status"]]

Psi diagnostics:
{'n': 134, 'psi_sd_h': 0.24433526018009394, 'psi_mean_h': 8.781607488954887, 'separation_supported': False, 'message': 'psi hardly varies: separation of circadian and wake-dependent effects is not supported'}


,date,day_type,clock,status
0,2026-03-10,workday,08:19,ok
1,2026-03-10,workday,13:19,ok
2,2026-03-10,workday,18:19,ok
3,2026-03-14,free,09:49,ok
4,2026-03-14,free,14:49,ok
5,2026-03-14,free,19:49,ok


## 6. Notebook location

This notebook is saved as `chronoalign_example.ipynb` in the repository root. Run the cells from top to bottom with the `/Users/arahjou/Documents/Com_Conda/.conda` Python environment selected.